# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



This task is a multiclass classification problem because the objective is to predict the content trend category.

Logistic Regression was selected as a simple and interpretable baseline model.

Random Forest was selected as a more powerful ensemble model because it captures nonlinear relationships, handles mixed feature types, and provides feature importance for interpretation.

Both models were trained using the same grouped train-test split and evaluated using the same performance metrics.




## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*



A grouped train-test split was used with **client_id** as the grouping variable.

This ensures that content from the same client does not appear in both the training and testing sets, reducing the risk of information leakage and providing a more realistic evaluation of model performance.

The dataset was split into **80% training** and **20% testing** using **GroupShuffleSplit** with **random_state = 42** to ensure reproducibility.

In [ ]:
# =====================================================
# Section 2 - Split Design
# =====================================================

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# -----------------------------------------------------
# Load Dataset
# -----------------------------------------------------

df = pd.read_csv("https://raw.githubusercontent.com/acecod3z/Flyrankinternship/main/data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

# -----------------------------------------------------
# Target Variable
# -----------------------------------------------------


y = df["trend_direction"]

# -----------------------------------------------------
# Remove Leakage Columns
# -----------------------------------------------------

drop_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

X = df.drop(columns=drop_columns)

# -----------------------------------------------------
# Grouped Train-Test Split
# -----------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        groups=df["client_id"]
    )
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("\nTrain Shape:", X_train.shape)
print("Test Shape :", X_test.shape)

print("\nTraining Label Distribution")
print(y_train.value_counts())

print("\nTesting Label Distribution")
print(y_test.value_counts())

Dataset Shape: (30000, 44)

Train Shape: (23837, 40)
Test Shape : (6163, 40)

Training Label Distribution
trend_direction
down      13113
stable     4661
up         3359
new        1934
flat        770
Name: count, dtype: int64

Testing Label Distribution
trend_direction
down      3149
stable    1301
up        1029
flat       382
new        302
Name: count, dtype: int64


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


Two machine learning models were trained:

- Logistic Regression
- Random Forest

Logistic Regression provides an interpretable linear baseline, while Random Forest captures nonlinear relationships and interactions between features.

Both models were trained and evaluated using the same grouped train-test split.

The Week-4 rule-based baseline ranked pages using search volume, CTR and average position.

Random Forest achieved the best overall performance among the trained models.

In [ ]:
#preprocessing
import pandas as pd

# ----------------------------
# Numeric columns
# ----------------------------

numeric_cols = X_train.select_dtypes(include=["number"]).columns

train_medians = X_train[numeric_cols].median()

X_train[numeric_cols] = X_train[numeric_cols].fillna(train_medians)
X_test[numeric_cols] = X_test[numeric_cols].fillna(train_medians)

# ----------------------------
# Categorical columns
# ----------------------------

categorical_cols = X_train.select_dtypes(include=["object"]).columns

X_train[categorical_cols] = X_train[categorical_cols].fillna("Unknown")
X_test[categorical_cols] = X_test[categorical_cols].fillna("Unknown")

# ----------------------------
# One-hot encode
# ----------------------------

X_train = pd.get_dummies(
    X_train,
    dummy_na=False
)

X_test = pd.get_dummies(
    X_test,
    dummy_na=False
)

# ----------------------------
# Match columns
# ----------------------------

X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

print("Remaining NaNs (Train):", X_train.isna().sum().sum())
print("Remaining NaNs (Test):", X_test.isna().sum().sum())


from sklearn.preprocessing import StandardScaler

# ----------------------------
# Scale numeric columns only
# ----------------------------

numeric_cols = X_train.select_dtypes(include=["number"]).columns

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print("Scaling complete.")

from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=5000,
    random_state=42,
    solver="lbfgs"
)

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)

print("Logistic Regression trained successfully.")

from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("Random Forest trained successfully!")

#evaluation

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

def evaluate(name, y_true, pred):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, average="weighted"),
        "Recall": recall_score(y_true, pred, average="weighted"),
        "F1 Score": f1_score(y_true, pred, average="weighted")
    }

results = pd.DataFrame([
    evaluate("Logistic Regression", y_test, log_pred),
    evaluate("Random Forest", y_test, rf_pred)
])

display(results)

print("\nRandom Forest Classification Report\n")
print(classification_report(y_test, rf_pred))

Remaining NaNs (Train): 0
Remaining NaNs (Test): 0
Scaling complete.
Logistic Regression trained successfully.
Random Forest trained successfully!


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.635080,0.644427,0.635080,0.618821
1,Random Forest,0.695603,0.673345,0.695603,0.657609



Random Forest Classification Report

              precision    recall  f1-score   support

        down       0.68      0.93      0.79      3149
        flat       1.00      1.00      1.00       382
         new       1.00      1.00      1.00       302
      stable       0.50      0.25      0.34      1301
          up       0.65      0.33      0.44      1029

    accuracy                           0.70      6163
   macro avg       0.77      0.70      0.71      6163
weighted avg       0.67      0.70      0.66      6163



### Baseline Comparison

The Week-4 baseline produced a ranked priority list for content review using a transparent rule-based scoring approach.

Because the baseline generated rankings rather than class predictions, direct Accuracy and F1 comparisons were not computed.

The machine learning models provide predictive performance that complements the rule-based baseline.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



Random Forest outperformed Logistic Regression across all evaluation metrics.

Most prediction errors occurred between the **stable** and **up** classes because these categories have similar behavioural patterns.

The **down** class achieved the highest recall, showing that declining pages have stronger predictive signals.

The **flat** and **new** classes were classified accurately because they contain more distinctive characteristics.

Overall, the model performs well on clear traffic trends but struggles when multiple trend categories share similar engagement patterns.

## Feature Interpretation

The Random Forest model identified the following features as the most important:

- impressions_prev_30d
- impressions_last_30d
- impressions_90d
- avg_position
- days_with_impressions

These features are reasonable because they capture search visibility, historical performance and user engagement, which are directly related to changes in content trends.

No suspicious feature dominated the model, suggesting that obvious feature leakage was avoided.

In [ ]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

display(importance.head(20))

,Feature,Importance
18,impressions_prev_30d,0.145816
15,impressions_last_30d,0.123962
5,impressions_90d,0.062329
25,avg_position,0.061325
13,days_with_impressions,0.054781
21,content_age_days,0.033123
17,sessions_last_30d,0.027672
20,sessions_prev_30d,0.024674
24,ctr,0.024578
7,pageviews_90d,0.023521


In [ ]:
#error analysis
errors = X_test.copy()

errors["Actual"] = y_test.values
errors["Predicted"] = rf_pred

wrong = errors[errors["Actual"] != errors["Predicted"]]

print(f"Total incorrect predictions: {len(wrong)}")

display(wrong.head(3))

Total incorrect predictions: 1876


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,impression_tier_good,impression_tier_low,impression_tier_moderate,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3,Actual,Predicted
13,-0.082816,-0.488045,-0.224854,-1.327087,-1.327024,-0.309573,-0.222002,-0.328486,-0.334778,-0.333936,...,False,False,True,False,False,True,False,False,stable,down
26,-0.089306,-0.488045,-0.224854,-0.342449,-0.409865,-0.182509,-0.183322,-0.298552,-0.291943,-0.289755,...,False,False,True,False,False,True,False,False,stable,down
36,-0.089306,-0.488045,-0.224854,-0.471390,-0.584938,-0.305735,-0.157536,-0.316513,-0.326211,-0.325100,...,False,False,True,False,True,False,False,False,stable,down


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.